# Práctica 1 · Paso 1 — Climatología 1995–2014 con dos reanálisis

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P1-01 · Versión 5.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_01_climatologia.ipynb)

**Objetivo.** Calcular la climatología mensual de su zona para las 9 variables en el periodo de referencia del protocolo común, y comparar ERA5 con CORe.

**Producto.** Tabla de doce meses por variable, climograma, panel de humedad, presión y viento, y resumen. Primera pieza del capítulo 1 (paso 1).

**Requisito.** Haber ejecutado los cuadernos comunes `CR01_descarga_reanalisis` y `CR02_series_zona` con la misma zona.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Elija sus datos en el formulario de la celda **1** (listas desplegables a la derecha) y ejecute las demás en orden.

<details><summary>Secuencia y variables</summary>


| Orden | Cuaderno común (las dos carreras) | Qué hace | Deja listo en `Mi unidad/CC_datos/<zona>/` |
|---|---|---|---|
| C1 | [`CR01_descarga_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb) | Descarga ERA5 y CORe solo para su zona (opción A, con su cuenta) o los trae del release del curso (opción B) | Archivos NetCDF e inventario |
| C2 | [`CR02_series_zona`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR02_series_zona.ipynb) | Promedia sobre la zona, ponderando cada celda por su área dentro del límite | Series diaria y mensual |

| Orden | Ingeniería Ambiental · Práctica 1 | Qué hace | Deja listo en `Mi unidad/CC_Ambiental/practica-1/<zona>/` |
|---|---|---|---|
| 1 | [`IA1_01_climatologia`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_01_climatologia.ipynb) | Paso 1: climatología 1995–2014 de las 9 variables con los dos reanálisis | Tablas y figuras |
| 2 | [`IA1_02_estacion_vs_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_02_estacion_vs_reanalisis.ipynb) | Paso 2: contraste de una estación con ERA5 y CORe | Métricas y figuras |


| Variable | Columna | Unidad |
|---|---|---|
| Temperatura media, máxima y mínima a 2 m | `tmed`, `tmax`, `tmin` | °C |
| Precipitación | `pr` | mm/día · mm/mes |
| Humedad relativa a 2 m | `hr` | % |
| Humedad específica a 2 m | `q` | g/kg |
| Presión atmosférica en superficie | `ps` | hPa |
| Viento a 10 m: componentes, velocidad y dirección de procedencia | `u10`, `v10`, `ws`, `wd` | m/s · ° |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante

In [ ]:
# @title Datos del estudiante y zona de trabajo (la misma en todos los cuadernos)
ESTUDIANTE = "Nombre Apellido"  # @param {type:"string"}
NIVEL = "provincia"  # @param ["provincia", "cantón"]
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
CANTON = "Manabí · Portoviejo"  # @param ["Azuay · Camilo Ponce Enriquez", "Azuay · Chordeleg", "Azuay · Cuenca", "Azuay · El Pan", "Azuay · Giron", "Azuay · Guachapala", "Azuay · Gualaceo", "Azuay · Nabon", "Azuay · Oña", "Azuay · Paute", "Azuay · Pucara", "Azuay · San Fernando", "Azuay · Santa Isabel", "Azuay · Sevilla De Oro", "Azuay · Sigsig", "Bolívar · Caluma", "Bolívar · Chillanes", "Bolívar · Chimbo", "Bolívar · Echeandia", "Bolívar · Guaranda", "Bolívar · Las Naves", "Bolívar · San Miguel", "Cañar · Azogues", "Cañar · Biblian", "Cañar · Cañar", "Cañar · Deleg", "Cañar · El Tambo", "Cañar · La Troncal", "Cañar · Suscal", "Carchi · Bolivar", "Carchi · Espejo", "Carchi · Mira", "Carchi · Montufar", "Carchi · San Pedro De Huaca", "Carchi · Tulcan", "Chimborazo · Alausi", "Chimborazo · Chambo", "Chimborazo · Chunchi", "Chimborazo · Colta", "Chimborazo · Cumanda", "Chimborazo · Guamote", "Chimborazo · Guano", "Chimborazo · Pallatanga", "Chimborazo · Penipe", "Chimborazo · Riobamba", "Cotopaxi · La Mana", "Cotopaxi · Latacunga", "Cotopaxi · Pangua", "Cotopaxi · Pujili", "Cotopaxi · Salcedo", "Cotopaxi · Saquisili", "Cotopaxi · Sigchos", "El Oro · Arenillas", "El Oro · Atahualpa", "El Oro · Balsas", "El Oro · Chilla", "El Oro · El Guabo", "El Oro · Huaquillas", "El Oro · Las Lajas", "El Oro · Machala", "El Oro · Marcabeli", "El Oro · Pasaje", "El Oro · Piñas", "El Oro · Portovelo", "El Oro · Santa Rosa", "El Oro · Zaruma", "Esmeraldas · Atacames", "Esmeraldas · Eloy Alfaro", "Esmeraldas · Esmeraldas", "Esmeraldas · La Concordia", "Esmeraldas · Muisne", "Esmeraldas · Quininde", "Esmeraldas · Rioverde", "Esmeraldas · San Lorenzo", "Galápagos · Isabela", "Galápagos · San Cristobal", "Galápagos · Santa Cruz", "Guayas · Alfredo Baquerizo Moreno", "Guayas · Balao", "Guayas · Balzar", "Guayas · Colimes", "Guayas · Crnel. Marcelino Maridueña", "Guayas · Daule", "Guayas · Duran", "Guayas · El Triunfo", "Guayas · Empalme", "Guayas · Gnral. Antonio Elizalde", "Guayas · Guayaquil", "Guayas · Isidro Ayora", "Guayas · Lomas De Sargentillo", "Guayas · Milagro", "Guayas · Naranjal", "Guayas · Naranjito", "Guayas · Nobol", "Guayas · Palestina", "Guayas · Pedro Carbo", "Guayas · Playas", "Guayas · Salitre", "Guayas · Samborondon", "Guayas · San Jacinto De Yaguachi", "Guayas · Santa Lucia", "Guayas · Simon Bolivar", "Imbabura · Antonio Ante", "Imbabura · Cotacachi", "Imbabura · Ibarra", "Imbabura · Otavalo", "Imbabura · Pimampiro", "Imbabura · San Miguel De Urcuqui", "Loja · Calvas", "Loja · Catamayo", "Loja · Celica", "Loja · Chaguarpamba", "Loja · Espindola", "Loja · Gonzanama", "Loja · Loja", "Loja · Macara", "Loja · Olmedo", "Loja · Paltas", "Loja · Pindal", "Loja · Puyango", "Loja · Quilanga", "Loja · Saraguro", "Loja · Sozoranga", "Loja · Zapotillo", "Los Ríos · Baba", "Los Ríos · Babahoyo", "Los Ríos · Buena Fe", "Los Ríos · Mocache", "Los Ríos · Montalvo", "Los Ríos · Palenque", "Los Ríos · Puebloviejo", "Los Ríos · Quevedo", "Los Ríos · Quinsaloma", "Los Ríos · Urdaneta", "Los Ríos · Valencia", "Los Ríos · Ventanas", "Los Ríos · Vinces", "Manabí · 24 De Mayo", "Manabí · Bolivar", "Manabí · Chone", "Manabí · El Carmen", "Manabí · Flavio Alfaro", "Manabí · Jama", "Manabí · Jaramijo", "Manabí · Jipijapa", "Manabí · Junin", "Manabí · Manta", "Manabí · Montecristi", "Manabí · Olmedo", "Manabí · Pajan", "Manabí · Pedernales", "Manabí · Pichincha", "Manabí · Portoviejo", "Manabí · Puerto Lopez", "Manabí · Rocafuerte", "Manabí · San Vicente", "Manabí · Santa Ana", "Manabí · Sucre", "Manabí · Tosagua", "Morona Santiago · Gualaquiza", "Morona Santiago · Huamboya", "Morona Santiago · Limon Indanza", "Morona Santiago · Logroðo", "Morona Santiago · Morona", "Morona Santiago · Pablo Sexto", "Morona Santiago · Palora", "Morona Santiago · San Juan Bosco", "Morona Santiago · Santiago", "Morona Santiago · Sucua", "Morona Santiago · Taisha", "Morona Santiago · Tiwintza", "Napo · Archidona", "Napo · Carlos Julio Arosemena Tola", "Napo · El Chaco", "Napo · Quijos", "Napo · Tena", "Orellana · Aguarico", "Orellana · La Joya De Los Sachas", "Orellana · Loreto", "Orellana · Orellana", "Pastaza · Arajuno", "Pastaza · Mera", "Pastaza · Pastaza", "Pastaza · Santa Clara", "Pichincha · Cayambe", "Pichincha · Mejia", "Pichincha · Pedro Moncayo", "Pichincha · Pedro Vicente Maldonado", "Pichincha · Puerto Quito", "Pichincha · Quito", "Pichincha · Rumiðahui", "Pichincha · San Miguel De Los Bancos", "Santa Elena · La Libertad", "Santa Elena · Salinas", "Santa Elena · Santa Elena", "Santo Domingo de los Tsáchilas · Santo Domingo", "Sucumbíos · Cascales", "Sucumbíos · Cuyabeno", "Sucumbíos · Gonzalo Pizarro", "Sucumbíos · Lago Agrio", "Sucumbíos · Putumayo", "Sucumbíos · Shushufindi", "Sucumbíos · Sucumbios", "Tungurahua · Ambato", "Tungurahua · Baños De Agua Santa", "Tungurahua · Cevallos", "Tungurahua · Mocha", "Tungurahua · Patate", "Tungurahua · Quero", "Tungurahua · San Pedro De Pelileo", "Tungurahua · Santiago De Pillaro", "Tungurahua · Tisaleo", "Zamora Chinchipe · Centinela Del Condor", "Zamora Chinchipe · Chinchipe", "Zamora Chinchipe · El Pangui", "Zamora Chinchipe · Nangaritza", "Zamora Chinchipe · Palanda", "Zamora Chinchipe · Paquisha", "Zamora Chinchipe · Yacuambi", "Zamora Chinchipe · Yantzaza", "Zamora Chinchipe · Zamora"]
# NIVEL = "provincia": se usa PROVINCIA y se ignora CANTON.
# NIVEL = "cantón": se usa CANTON (la lista está ordenada por provincia) y su provincia.
if NIVEL == "cantón":
    PROVINCIA, CANTON = CANTON.split(" · ")
else:
    CANTON = None

REF_INI, REF_FIN = 1995, 2014   # periodo de referencia del protocolo común: no cambiar

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos y series comunes a las dos carreras (CR01, CR02)
RES = BASE / "CC_Ambiental" / "practica-1" / S                        # productos de este cuaderno
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

## 3. Serie mensual del periodo de referencia

In [ ]:
M = pd.read_csv(DATOS / f"serie_mensual_{S}.csv", index_col="fecha", parse_dates=True).loc[f"{REF_INI}":f"{REF_FIN}"]
FUENTES = [f for f in ("ERA5", "CORe") if f"{f}_tmed" in M]
assert len(M) == (REF_FIN - REF_INI + 1) * 12, "La serie no cubre completo el periodo de referencia"
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
print("Fuentes:", FUENTES, "·", len(M), "meses")

## 4. Climatología mensual

Para cada mes se promedian los veinte años; la desviación estándar mide la variabilidad entre años. La velocidad del viento es el promedio de las velocidades; la dirección, la del viento medio vectorial (u, v).

In [ ]:
g = M.groupby(M.index.month)
MEDIA, DESV = g.mean(), g.std()
for f in FUENTES:
    if f"{f}_u10" in MEDIA:
        ws_vec, MEDIA[f"{f}_wd"] = V.viento(MEDIA[f"{f}_u10"], MEDIA[f"{f}_v10"])
        if f"{f}_ws" not in MEDIA: MEDIA[f"{f}_ws"] = ws_vec
MEDIA.index = DESV.index = MESES
def tabla(v): return MEDIA[[f"{f}_{v}" for f in FUENTES if f"{f}_{v}" in MEDIA]].round(1)
for v in ("tmed", "tmax", "tmin", "pr", "hr", "q", "ps", "ws", "wd"):
    print(f"--- {V.NOMBRES[v]} ({V.UNIDADES[v] if v != 'pr' else 'mm/mes'})"); display(tabla(v).T)

In [ ]:
def resumen(f):
    c = MEDIA; anual = M[f"{f}_pr"].groupby(M.index.year).sum()
    u, v = M[f"{f}_u10"].mean(), M[f"{f}_v10"].mean(); ws_vec, wd = V.viento(u, v)
    ws = M[f"{f}_ws"].mean() if f"{f}_ws" in M else ws_vec
    rumbos = ["N", "NE", "E", "SE", "S", "SO", "O", "NO"]
    r = {"T media anual (°C)": round(M[f"{f}_tmed"].mean(), 2),
            "Tmax media (°C)": round(M[f"{f}_tmax"].mean(), 2) if f"{f}_tmax" in M else None,
            "Tmin media (°C)": round(M[f"{f}_tmin"].mean(), 2) if f"{f}_tmin" in M else None,
            "Amplitud térmica anual (°C)": round(c[f"{f}_tmed"].max() - c[f"{f}_tmed"].min(), 2),
            "P anual media (mm)": round(anual.mean(), 0), "P anual, desviación (mm)": round(anual.std(), 0),
            "Mes más lluvioso": c[f"{f}_pr"].idxmax(), "Mes más seco": c[f"{f}_pr"].idxmin(),
            "% de lluvia dic–may": round(100 * c.loc[["Dic", "Ene", "Feb", "Mar", "Abr", "May"], f"{f}_pr"].sum() / c[f"{f}_pr"].sum(), 1),
            "HR media (%)": round(M[f"{f}_hr"].mean(), 1), "q media (g/kg)": round(M[f"{f}_q"].mean(), 2),
            "Presión media (hPa)": round(M[f"{f}_ps"].mean(), 1),
            "Viento medio (m/s)": round(float(ws), 2), "Dirección predominante": f"{rumbos[int(((float(wd) + 22.5) % 360) // 45)]} ({float(wd):.0f}°)"}
    return r
RESUMEN = pd.DataFrame({f: resumen(f) for f in FUENTES})
RESUMEN

## 5. Climograma comparado

In [ ]:
x = np.arange(12); w = 0.8 / len(FUENTES)
col = {"ERA5": ("#4C78A8", "#C2531F"), "CORe": ("#9ECAE9", "#F2A65A")}
fig, ax1 = plt.subplots(figsize=(10, 5))
for i, f in enumerate(FUENTES):
    ax1.bar(x - 0.4 + w * (i + 0.5), MEDIA[f"{f}_pr"], w, yerr=DESV[f"{f}_pr"], capsize=2, color=col[f][0], label=f"P · {f}")
ax1.set_xticks(x, MESES); ax1.set_ylabel("Precipitación (mm/mes)"); ax1.set_ylim(bottom=0)
ax2 = ax1.twinx()
for f in FUENTES:
    ax2.plot(x, MEDIA[f"{f}_tmed"], "o-" if f == "ERA5" else "s--", color=col[f][1], lw=2, label=f"T · {f}")
    if f"{f}_tmax" in MEDIA:
        ax2.fill_between(x, MEDIA[f"{f}_tmin"], MEDIA[f"{f}_tmax"], color=col[f][1], alpha=0.12)
ax2.set_ylabel("Temperatura (°C) · banda: Tmin–Tmax")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88), fontsize=8)
ax1.set_title(f"Climograma de {Z['nombre']}, {REF_INI}–{REF_FIN}")
fig.text(0.01, 0.01, f"Fuentes: ERA5 (Hersbach et al., 2020); CORe (NOAA/NCEP). Elaboración: {ESTUDIANTE}.", fontsize=7)
fig.tight_layout(); fig.savefig(RES / f"IA1_01_climograma_{S}.png", dpi=300); plt.show()

## 6. Humedad, presión y viento

La **humedad específica** dice cuánto vapor de agua hay realmente en el aire (gramos por kilogramo); la **relativa**, qué tan cerca está de saturarse, y por eso depende también de la temperatura. Si la humedad relativa sube en la estación seca mientras la específica baja, el aire no tiene más vapor: solo está más frío. Las flechas indican hacia dónde sopla el viento medio de cada mes; su largo, la velocidad.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
for f in FUENTES:
    c = col[f][1]
    ax[0].plot(x, MEDIA[f"{f}_q"], "o-", color=c, label=f); ax[1].plot(x, MEDIA[f"{f}_hr"], "o-", color=c, label=f)
    ax[2].plot(x, MEDIA[f"{f}_ps"], "o-", color=c, label=f); ax[3].plot(x, MEDIA[f"{f}_ws"], "o-", color=c, label=f)
    ax[3].quiver(x, MEDIA[f"{f}_ws"], MEDIA[f"{f}_u10"], MEDIA[f"{f}_v10"], color=c, scale=25, width=0.006)
for a, t in zip(ax, ("Humedad específica (g/kg)", "Humedad relativa (%)", "Presión en superficie (hPa)", "Viento medio a 10 m (m/s)")):
    a.set_xticks(x, MESES, fontsize=8); a.set_title(t); a.legend(fontsize=8)
fig.suptitle(f"{Z['nombre']}, {REF_INI}–{REF_FIN}"); fig.tight_layout()
fig.savefig(RES / f"IA1_01_humedad_presion_viento_{S}.png", dpi=300); plt.show()

## 7. ¿Cuánto difieren los dos reanálisis?

La diferencia mes a mes (CORe − ERA5) es una medida de la **incertidumbre observacional**. La presión de CORe puede diferir varios hPa solo porque su terreno, más grueso, tiene otra altura.

In [ ]:
if len(FUENTES) == 2:
    DIF = pd.DataFrame({v: (MEDIA[f"CORe_{v}"] - MEDIA[f"ERA5_{v}"]) for v in ("tmed", "tmax", "tmin", "pr", "hr", "q", "ps", "ws")
                        if f"CORe_{v}" in MEDIA and f"ERA5_{v}" in MEDIA}).round(2)
    DIF["pr_%"] = (100 * DIF["pr"] / MEDIA["ERA5_pr"]).round(1)
    DIF["wd"] = ((MEDIA["CORe_wd"] - MEDIA["ERA5_wd"] + 180) % 360 - 180).round(0)
    display(DIF)
else:
    DIF = None; print("Solo hay una fuente disponible.")

## 8. Comprobaciones y guardado

In [ ]:
problemas = []
for f in FUENTES:
    r = RESUMEN[f]
    if not 5 <= r["T media anual (°C)"] <= 32: problemas.append(f"{f}: temperatura fuera de rango")
    if not 50 <= r["P anual media (mm)"] <= 6000: problemas.append(f"{f}: precipitación fuera de rango")
    if not 30 <= r["HR media (%)"] <= 100: problemas.append(f"{f}: humedad relativa fuera de rango")
    if not 1 <= r["q media (g/kg)"] <= 25: problemas.append(f"{f}: humedad específica fuera de rango")
    if not 500 <= r["Presión media (hPa)"] <= 1050: problemas.append(f"{f}: presión fuera de rango")
print("Sin problemas." if not problemas else "\n".join(problemas))
pd.concat({"media": MEDIA, "desv": DESV}, axis=1).round(3).to_csv(RES / f"IA1_01_climatologia_{S}_{REF_INI}_{REF_FIN}.csv", encoding="utf-8")
RESUMEN.to_csv(RES / f"IA1_01_resumen_{S}.csv", encoding="utf-8")
if DIF is not None: DIF.to_csv(RES / f"IA1_01_diferencias_{S}.csv", encoding="utf-8")
print(sorted(p.name for p in RES.glob("IA1_01_*")))

## 9. Preguntas para el capítulo 1

1. Describa el clima de su zona con los números del resumen: temperatura, amplitud térmica, lluvia anual y su estacionalidad, humedad y viento predominante.
2. ¿Cómo cambian la humedad específica, la relativa y el viento entre la estación lluviosa y la seca? ¿Se mueven igual las dos humedades? Explíquelo con la temperatura y relaciónelo con el desplazamiento de la zona de convergencia intertropical.
3. ¿En qué variables difieren más ERA5 y CORe? ¿Por qué la presión es la que más depende de la resolución?
4. Exprese la precipitación anual de su zona como un rango que incluya a las dos fuentes.

**Siguiente:** `IA1_02_estacion_vs_reanalisis`.